# 🎓 U.S. Student Loan Risk Analysis
## Which Schools Leave Students in Debt?

**Part 2** of Higher Education Analysis Series
- Part 1 (Python): [College Tuition Spending Analysis](https://github.com/Katherine-code-web/college-tuition-analysis) — *Where does tuition money go?*
- **Part 2 (SQL): This notebook** — *What's the cost to students, and who's at risk?*

---
### 📋 How to Use This Notebook
1. **Step 1**: Run the Setup cells to download data and build the database
2. **Step 2**: Run each SQL query section and review results
3. **Step 3**: Write your insights in the provided spaces
4. **Step 4**: Export results for your GitHub README

> 💡 All SQL runs inside this Colab notebook using SQLite — no external tools needed!

---
## ⚙️ Step 1: Environment Setup
Run the cells below **in order** to set everything up.

### 1.1 Install & Import Libraries

In [ ]:
# Install (if needed) and import
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

# Helper function: run SQL and return DataFrame
def run_sql(query, db_path='student_loans.db'):
    """Execute SQL query and return results as DataFrame."""
    conn = sqlite3.connect(db_path)
    df = pd.read_sql_query(query, conn)
    conn.close()
    return df

# Helper: run SQL and display nicely
def sql(query, db_path='student_loans.db'):
    """Run SQL, display results, and return DataFrame."""
    df = run_sql(query, db_path)
    print(f'📊 {len(df)} rows returned')
    display(df)
    return df

print('✅ Libraries loaded!')

### 1.2 Download College Scorecard Data

We'll download directly from the U.S. Department of Education.

> ⚠️ The file is ~170MB — it may take 1-2 minutes to download.

In [ ]:
import os, zipfile, glob

# Download College Scorecard data
DATA_URL = 'https://ed-public-download.app.cloud.gov/downloads/Most-Recent-Cohorts-Institution_04192023.zip'
ZIP_FILE = 'scorecard.zip'
CSV_FILE = None

if not glob.glob('Most-Recent*.csv') and not os.path.exists('scorecard.csv'):
    print('📥 Downloading College Scorecard data...')
    !wget -q --show-progress -O {ZIP_FILE} "{DATA_URL}"
    
    print('📦 Extracting...')
    with zipfile.ZipFile(ZIP_FILE, 'r') as z:
        z.extractall('.')
    os.remove(ZIP_FILE)

# Find the CSV file
csv_files = glob.glob('Most-Recent*.csv')
if csv_files:
    CSV_FILE = csv_files[0]
elif os.path.exists('scorecard.csv'):
    CSV_FILE = 'scorecard.csv'

if CSV_FILE:
    file_size = os.path.getsize(CSV_FILE) / (1024*1024)
    print(f'✅ Data ready: {CSV_FILE} ({file_size:.1f} MB)')
else:
    print('❌ Could not find data file. Please upload manually.')

#### 🔄 Alternative: Manual Upload
If the download above fails, you can:
1. Go to https://collegescorecard.ed.gov/data
2. Download "Most Recent Institution-Level Data"
3. Upload the CSV using the cell below

In [ ]:
# ONLY run this if the download above failed
# from google.colab import files
# uploaded = files.upload()  # Upload your CSV here
# CSV_FILE = list(uploaded.keys())[0]
# print(f'✅ Uploaded: {CSV_FILE}')

### 1.3 Build the SQLite Database

This cell reads the CSV and creates 3 normalized tables:
- `institutions` — school info (name, state, type)
- `institution_costs` — tuition, financial aid
- `loan_outcomes` — debt, default rates, earnings

In [ ]:
# ============================================
# Build SQLite Database from College Scorecard
# ============================================

DB_PATH = 'student_loans.db'

# State → Region mapping
REGION_MAP = {
    'CT':'Northeast','ME':'Northeast','MA':'Northeast','NH':'Northeast',
    'RI':'Northeast','VT':'Northeast','NJ':'Northeast','NY':'Northeast','PA':'Northeast',
    'IL':'Midwest','IN':'Midwest','MI':'Midwest','OH':'Midwest','WI':'Midwest',
    'IA':'Midwest','KS':'Midwest','MN':'Midwest','MO':'Midwest',
    'NE':'Midwest','ND':'Midwest','SD':'Midwest',
    'DE':'South','FL':'South','GA':'South','MD':'South','NC':'South',
    'SC':'South','VA':'South','DC':'South','WV':'South',
    'AL':'South','KY':'South','MS':'South','TN':'South',
    'AR':'South','LA':'South','OK':'South','TX':'South',
    'AZ':'West','CO':'West','ID':'West','MT':'West','NV':'West',
    'NM':'West','UT':'West','WY':'West','AK':'West','CA':'West',
    'HI':'West','OR':'West','WA':'West',
}
CONTROL_LABELS = {1:'Public', 2:'Private Non-Profit', 3:'Private For-Profit'}
DEGREE_LABELS = {0:'Not classified', 1:'Certificate', 2:"Associate's", 3:"Bachelor's", 4:'Graduate'}

print(f'📖 Reading {CSV_FILE}...')
df_raw = pd.read_csv(CSV_FILE, low_memory=False, na_values=['NULL','PrivacySuppressed'])
df_raw.columns = df_raw.columns.str.upper()
print(f'   {len(df_raw)} institutions, {len(df_raw.columns)} columns')

# Select and clean columns
cols = {
    'UNITID':'unitid','INSTNM':'inst_name','STABBR':'state','CONTROL':'control',
    'PREDDEG':'pred_degree','ADM_RATE':'adm_rate','UGDS':'ugds',
    'SATVRMID':'sat_verbal_mid','SATMTMID':'sat_math_mid',
    'COSTT4_A':'cost_attendance','TUITIONFEE_IN':'tuition_in','TUITIONFEE_OUT':'tuition_out',
    'PCTFLOAN':'pct_fed_loan','PCTPELL':'pct_pell',
    'NPT4_PUB':'net_price_pub','NPT4_PRIV':'net_price_priv',
    'DEBT_MDN':'debt_median','GRAD_DEBT_MDN':'grad_debt_median',
    'CDR3':'default_rate_3yr','RPY_3YR_RT_SUPP':'repayment_rate_3yr','RPY_5YR_RT_SUPP':'repayment_rate_5yr',
    'MD_EARN_WNE_P6':'earnings_6yr','MD_EARN_WNE_P10':'earnings_10yr',
    'C150_4':'completion_rate',
}
available = {k:v for k,v in cols.items() if k in df_raw.columns}
df = df_raw[list(available.keys())].rename(columns=available)

# Convert numeric
num_cols = ['adm_rate','ugds','sat_verbal_mid','sat_math_mid','cost_attendance',
            'tuition_in','tuition_out','pct_fed_loan','pct_pell',
            'debt_median','grad_debt_median','default_rate_3yr',
            'repayment_rate_3yr','repayment_rate_5yr',
            'earnings_6yr','earnings_10yr','completion_rate']
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

# Derived columns
df['control'] = pd.to_numeric(df['control'], errors='coerce')
df['control_label'] = df['control'].map(CONTROL_LABELS)
df['pred_degree'] = pd.to_numeric(df['pred_degree'], errors='coerce')
df['degree_label'] = df['pred_degree'].map(DEGREE_LABELS)
df['region'] = df['state'].map(REGION_MAP)
if 'net_price_pub' in df.columns:
    df['avg_net_price'] = df['net_price_pub'].fillna(df.get('net_price_priv'))

# ---- Create SQLite Database ----
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
conn = sqlite3.connect(DB_PATH)

# Table 1: institutions
inst_cols = ['unitid','inst_name','state','control','control_label',
             'pred_degree','degree_label','adm_rate','ugds',
             'sat_verbal_mid','sat_math_mid','region']
df[[c for c in inst_cols if c in df.columns]].drop_duplicates('unitid').to_sql(
    'institutions', conn, if_exists='replace', index=False)

# Table 2: institution_costs
cost_cols = ['unitid','cost_attendance','tuition_in','tuition_out',
             'pct_fed_loan','pct_pell','avg_net_price']
df[[c for c in cost_cols if c in df.columns]].drop_duplicates('unitid').to_sql(
    'institution_costs', conn, if_exists='replace', index=False)

# Table 3: loan_outcomes
loan_cols = ['unitid','debt_median','grad_debt_median','default_rate_3yr',
             'repayment_rate_3yr','repayment_rate_5yr',
             'earnings_6yr','earnings_10yr','completion_rate']
df[[c for c in loan_cols if c in df.columns]].drop_duplicates('unitid').to_sql(
    'loan_outcomes', conn, if_exists='replace', index=False)

# Create indexes
conn.execute('CREATE INDEX IF NOT EXISTS idx_inst_state ON institutions(state)')
conn.execute('CREATE INDEX IF NOT EXISTS idx_inst_control ON institutions(control)')
conn.execute('CREATE INDEX IF NOT EXISTS idx_loan_default ON loan_outcomes(default_rate_3yr)')
conn.execute('CREATE INDEX IF NOT EXISTS idx_loan_earnings ON loan_outcomes(earnings_10yr)')

# Verify
print('\n📋 Database Summary:')
for tbl in ['institutions', 'institution_costs', 'loan_outcomes']:
    cnt = pd.read_sql(f'SELECT COUNT(*) as n FROM {tbl}', conn).iloc[0,0]
    print(f'   {tbl}: {cnt:,} rows')

conn.close()
print(f'\n✅ Database ready: {DB_PATH}')

### 1.4 Data Quality Check
驗證資料品質，確認 NULL 比例和分佈。

In [ ]:
# ============================================
# 03_data_cleaning.sql — Data Validation
# ============================================

print('📋 Table Row Counts:')
display(sql("""
SELECT 'institutions' AS table_name, COUNT(*) AS rows FROM institutions
UNION ALL SELECT 'institution_costs', COUNT(*) FROM institution_costs
UNION ALL SELECT 'loan_outcomes', COUNT(*) FROM loan_outcomes
"""))

print('\n📋 NULL Rates for Key Columns:')
display(sql("""
SELECT 
    COUNT(*) AS total,
    SUM(CASE WHEN debt_median IS NULL THEN 1 ELSE 0 END) AS null_debt,
    SUM(CASE WHEN default_rate_3yr IS NULL THEN 1 ELSE 0 END) AS null_default,
    SUM(CASE WHEN earnings_10yr IS NULL THEN 1 ELSE 0 END) AS null_earnings,
    SUM(CASE WHEN completion_rate IS NULL THEN 1 ELSE 0 END) AS null_completion,
    ROUND(100.0 * SUM(CASE WHEN debt_median IS NOT NULL 
        AND default_rate_3yr IS NOT NULL 
        AND earnings_10yr IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 1) 
        AS pct_complete_records
FROM loan_outcomes
"""))

print('\n📋 School Type Distribution:')
display(sql("""
SELECT 
    control_label, 
    COUNT(*) AS cnt,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM institutions), 1) AS pct
FROM institutions
GROUP BY control_label
ORDER BY cnt DESC
"""))

---
## 📊 Theme 1: Student Loan Landscape Overview
*"How much do students borrow across the U.S.?"*

### Q1: State-Level Loan Burden Ranking
**Skills**: `JOIN`, `RANK()` Window Function, `GROUP BY`, `HAVING`

**Question**: Which states have the highest median student debt?

In [ ]:
q1 = sql("""
SELECT 
    i.state,
    COUNT(*)                                    AS num_schools,
    ROUND(AVG(lo.debt_median), 0)               AS avg_median_debt,
    ROUND(AVG(lo.default_rate_3yr) * 100, 1)    AS avg_default_rate_pct,
    ROUND(AVG(lo.earnings_10yr), 0)             AS avg_earnings_10yr,
    RANK() OVER (ORDER BY AVG(lo.debt_median) DESC) AS debt_rank
FROM institutions i
JOIN loan_outcomes lo ON i.unitid = lo.unitid
WHERE lo.debt_median IS NOT NULL
GROUP BY i.state
HAVING COUNT(*) >= 10
ORDER BY avg_median_debt DESC
LIMIT 15
""")

In [ ]:
# 📈 Visualization: Top 15 States by Median Debt
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(q1['state'][::-1], q1['avg_median_debt'][::-1], color='#3498db')
ax.set_xlabel('Average Median Debt ($)', fontsize=12)
ax.set_title('Top 15 States by Average Student Debt', fontsize=14, fontweight='bold')
for bar, val in zip(bars, q1['avg_median_debt'][::-1]):
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
            f'${val:,.0f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('q1_state_debt_ranking.png', dpi=150, bbox_inches='tight')
plt.show()

#### 💡 Your Insight for Q1:
> *Write your observation here after running the query. For example:*
> - Which states rank highest/lowest?
> - Any surprising patterns?
> - Does high debt = high earnings?

### Q2: Public vs Private Non-Profit vs For-Profit
**Skills**: `JOIN` (3 tables), `GROUP BY`, `CASE WHEN` (implicit via ROUND)

**Question**: How do debt & outcomes differ by school type?

In [ ]:
q2 = sql("""
SELECT 
    i.control_label                                     AS school_type,
    COUNT(*)                                            AS num_schools,
    ROUND(AVG(lo.debt_median), 0)                       AS avg_debt,
    ROUND(AVG(lo.default_rate_3yr) * 100, 1)            AS avg_default_pct,
    ROUND(AVG(lo.repayment_rate_3yr) * 100, 1)          AS avg_repayment_pct,
    ROUND(AVG(lo.earnings_10yr), 0)                     AS avg_earnings_10yr,
    ROUND(AVG(lo.completion_rate) * 100, 1)             AS avg_completion_pct,
    ROUND(AVG(ic.pct_fed_loan) * 100, 1)                AS avg_pct_with_loans,
    ROUND(AVG(lo.debt_median) / NULLIF(AVG(lo.earnings_10yr), 0), 2) AS debt_to_earnings
FROM institutions i
JOIN loan_outcomes lo ON i.unitid = lo.unitid
JOIN institution_costs ic ON i.unitid = ic.unitid
WHERE lo.debt_median IS NOT NULL
GROUP BY i.control_label
ORDER BY avg_default_pct DESC
""")

In [ ]:
# 📈 Visualization: School Type Comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

colors = ['#2ecc71', '#3498db', '#e74c3c']
types = q2['school_type'].tolist()

axes[0].bar(types, q2['avg_debt'], color=colors)
axes[0].set_title('Average Debt', fontweight='bold')
axes[0].set_ylabel('$')
axes[0].tick_params(axis='x', rotation=15)

axes[1].bar(types, q2['avg_default_pct'], color=colors)
axes[1].set_title('Default Rate (%)', fontweight='bold')
axes[1].tick_params(axis='x', rotation=15)

axes[2].bar(types, q2['avg_earnings_10yr'], color=colors)
axes[2].set_title('Earnings (10yr)', fontweight='bold')
axes[2].set_ylabel('$')
axes[2].tick_params(axis='x', rotation=15)

plt.suptitle('Public vs Private: Debt, Default, & Earnings', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('q2_school_type_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

#### 💡 Your Insight for Q2:
> - Which school type has highest default rate?
> - Is private for-profit worth the cost?
> - Compare debt-to-earnings ratio across types

### Q3: Top 20 Schools with Highest Debt + Risk Flag
**Skills**: Multiple `JOIN`, `CASE WHEN` risk classification

**Question**: Which schools send students out with the most debt?

In [ ]:
q3 = sql("""
SELECT 
    i.inst_name,
    i.state,
    i.control_label,
    ROUND(lo.debt_median, 0)                    AS median_debt,
    ROUND(lo.default_rate_3yr * 100, 1)         AS default_rate_pct,
    ROUND(lo.earnings_10yr, 0)                  AS earnings_10yr,
    ROUND(lo.completion_rate * 100, 1)          AS completion_pct,
    ROUND(ic.cost_attendance, 0)                AS cost_attendance,
    CASE 
        WHEN lo.debt_median > 30000 AND lo.earnings_10yr < 40000 
            THEN '⚠️ HIGH RISK'
        WHEN lo.debt_median > 25000 AND lo.earnings_10yr < 50000 
            THEN '⚡ MODERATE'
        ELSE '✅ OK'
    END AS risk_flag
FROM institutions i
JOIN loan_outcomes lo ON i.unitid = lo.unitid
JOIN institution_costs ic ON i.unitid = ic.unitid
WHERE lo.debt_median IS NOT NULL
  AND lo.earnings_10yr IS NOT NULL
  AND i.ugds >= 500
ORDER BY lo.debt_median DESC
LIMIT 20
""")

---
## 🔴 Theme 2: Default Risk Deep Dive
*"Who is at risk, and what do high-risk schools have in common?"*

### Q4: Risk Tier Classification System
**Skills**: `CTE`, `NTILE()`, `CASE WHEN`, composite scoring

**Business Value**: Build a multi-factor risk scoring framework

In [ ]:
q4 = sql("""
WITH risk_scored AS (
    SELECT 
        i.unitid, i.inst_name, i.state, i.control_label,
        lo.default_rate_3yr, lo.repayment_rate_3yr,
        lo.completion_rate, lo.earnings_10yr, lo.debt_median,
        NTILE(5) OVER (ORDER BY lo.default_rate_3yr DESC)    AS default_score,
        NTILE(5) OVER (ORDER BY lo.repayment_rate_3yr ASC)   AS repayment_score,
        NTILE(5) OVER (ORDER BY lo.completion_rate ASC)       AS completion_score,
        NTILE(5) OVER (ORDER BY lo.earnings_10yr ASC)         AS earnings_score
    FROM institutions i
    JOIN loan_outcomes lo ON i.unitid = lo.unitid
    WHERE lo.default_rate_3yr IS NOT NULL
      AND lo.repayment_rate_3yr IS NOT NULL
      AND lo.completion_rate IS NOT NULL
      AND lo.earnings_10yr IS NOT NULL
),
risk_tiered AS (
    SELECT *,
        (default_score + repayment_score + completion_score + earnings_score) AS composite_score,
        CASE 
            WHEN (default_score + repayment_score + completion_score + earnings_score) >= 16 THEN '🔴 High Risk'
            WHEN (default_score + repayment_score + completion_score + earnings_score) >= 12 THEN '🟡 Moderate Risk'
            WHEN (default_score + repayment_score + completion_score + earnings_score) >= 8  THEN '🟢 Low Risk'
            ELSE '🔵 Very Safe'
        END AS risk_tier
    FROM risk_scored
)
SELECT 
    risk_tier,
    COUNT(*) AS num_schools,
    ROUND(AVG(default_rate_3yr) * 100, 1) AS avg_default_pct,
    ROUND(AVG(repayment_rate_3yr) * 100, 1) AS avg_repayment_pct,
    ROUND(AVG(completion_rate) * 100, 1) AS avg_completion_pct,
    ROUND(AVG(earnings_10yr), 0) AS avg_earnings,
    ROUND(AVG(debt_median), 0) AS avg_debt
FROM risk_tiered
GROUP BY risk_tier
ORDER BY avg_default_pct DESC
""")

In [ ]:
# 📈 Visualization: Risk Tier Comparison
fig, ax = plt.subplots(figsize=(10, 5))
tiers = q4['risk_tier'].tolist()
tier_colors = ['#e74c3c', '#f39c12', '#2ecc71', '#3498db']
x = range(len(tiers))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], q4['avg_default_pct'], width, label='Default Rate %', color='#e74c3c', alpha=0.8)
bars2 = ax.bar([i + width/2 for i in x], q4['avg_completion_pct'], width, label='Completion Rate %', color='#2ecc71', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(tiers, fontsize=10)
ax.set_ylabel('Percentage (%)')
ax.set_title('Risk Tier: Default Rate vs Completion Rate', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('q4_risk_tiers.png', dpi=150, bbox_inches='tight')
plt.show()

#### 💡 Your Insight for Q4:
> - How big is the gap between high-risk and safe schools?
> - What's the earnings difference between tiers?
> - How many schools fall into each tier?

### Q5: What Do High-Default Schools Have in Common?
**Skills**: `CTE`, `CASE WHEN` grouping, `SUM(CASE...)` for composition

**Business Value**: Identify structural patterns behind risk

In [ ]:
q5 = sql("""
WITH school_groups AS (
    SELECT 
        i.*, lo.default_rate_3yr, lo.earnings_10yr,
        lo.debt_median, lo.completion_rate,
        ic.pct_fed_loan, ic.pct_pell, ic.cost_attendance,
        CASE 
            WHEN lo.default_rate_3yr >= 0.15 THEN 'High Default (≥15%)'
            WHEN lo.default_rate_3yr >= 0.07 THEN 'Medium Default (7-15%)'
            ELSE 'Low Default (<7%)'
        END AS default_group
    FROM institutions i
    JOIN loan_outcomes lo ON i.unitid = lo.unitid
    JOIN institution_costs ic ON i.unitid = ic.unitid
    WHERE lo.default_rate_3yr IS NOT NULL
)
SELECT 
    default_group,
    COUNT(*) AS num_schools,
    ROUND(100.0 * SUM(CASE WHEN control = 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_public,
    ROUND(100.0 * SUM(CASE WHEN control = 2 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_private_np,
    ROUND(100.0 * SUM(CASE WHEN control = 3 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_for_profit,
    ROUND(AVG(cost_attendance), 0) AS avg_cost,
    ROUND(AVG(debt_median), 0) AS avg_debt,
    ROUND(AVG(pct_pell) * 100, 1) AS avg_pct_pell,
    ROUND(AVG(earnings_10yr), 0) AS avg_earnings,
    ROUND(AVG(completion_rate) * 100, 1) AS avg_completion_pct
FROM school_groups
GROUP BY default_group
ORDER BY CASE default_group
    WHEN 'High Default (≥15%)' THEN 1
    WHEN 'Medium Default (7-15%)' THEN 2
    ELSE 3 END
""")

---
## 💰 Theme 3: Return on Investment
*"Is college worth it? Which schools give students the best return?"*

### Q6: College ROI Scorecard — Top 30 Best Value Schools
**Skills**: `CTE`, `NTILE()`, `RANK()`, composite scoring

**Business Value**: Actionable ranking for prospective students

In [ ]:
q6 = sql("""
WITH roi_metrics AS (
    SELECT 
        i.unitid, i.inst_name, i.state, i.control_label, i.degree_label, i.ugds,
        lo.debt_median, lo.earnings_10yr, lo.default_rate_3yr,
        lo.completion_rate, lo.repayment_rate_3yr, ic.cost_attendance,
        ROUND(lo.earnings_10yr / NULLIF(lo.debt_median, 0), 2) AS earnings_to_debt,
        ROUND(lo.earnings_10yr - lo.debt_median, 0) AS net_gain
    FROM institutions i
    JOIN loan_outcomes lo ON i.unitid = lo.unitid
    JOIN institution_costs ic ON i.unitid = ic.unitid
    WHERE lo.earnings_10yr IS NOT NULL AND lo.debt_median IS NOT NULL
      AND lo.debt_median > 0 AND i.ugds >= 500 AND i.pred_degree = 3
),
scored AS (
    SELECT *,
        NTILE(5) OVER (ORDER BY earnings_to_debt) AS roi_score,
        NTILE(5) OVER (ORDER BY default_rate_3yr DESC) AS safety_score,
        NTILE(5) OVER (ORDER BY completion_rate) AS comp_score,
        NTILE(5) OVER (ORDER BY repayment_rate_3yr) AS repay_score
    FROM roi_metrics
)
SELECT 
    RANK() OVER (ORDER BY (roi_score + safety_score + comp_score + repay_score) DESC) AS rank,
    inst_name, state, control_label,
    ROUND(debt_median, 0) AS debt,
    ROUND(earnings_10yr, 0) AS earnings_10yr,
    earnings_to_debt AS roi_ratio,
    ROUND(default_rate_3yr * 100, 1) AS default_pct,
    ROUND(completion_rate * 100, 1) AS completion_pct,
    (roi_score + safety_score + comp_score + repay_score) AS total_score
FROM scored
ORDER BY total_score DESC, earnings_to_debt DESC
LIMIT 30
""")

### Q7: Expensive vs Affordable Schools — Who Wins?
**Skills**: `CTE`, `NTILE()` quartiles, grouped comparison

**Question**: Do expensive schools actually produce better outcomes?

In [ ]:
q7 = sql("""
WITH cost_tiers AS (
    SELECT 
        i.unitid, i.inst_name, i.control_label,
        ic.cost_attendance, lo.debt_median, lo.earnings_10yr,
        lo.default_rate_3yr, lo.completion_rate,
        NTILE(4) OVER (ORDER BY ic.cost_attendance) AS cost_quartile
    FROM institutions i
    JOIN institution_costs ic ON i.unitid = ic.unitid
    JOIN loan_outcomes lo ON i.unitid = lo.unitid
    WHERE ic.cost_attendance IS NOT NULL
      AND lo.earnings_10yr IS NOT NULL AND lo.debt_median IS NOT NULL
      AND i.pred_degree = 3
)
SELECT 
    CASE cost_quartile
        WHEN 1 THEN 'Q1: Most Affordable'
        WHEN 2 THEN 'Q2: Below Avg Cost'
        WHEN 3 THEN 'Q3: Above Avg Cost'
        WHEN 4 THEN 'Q4: Most Expensive'
    END AS cost_tier,
    COUNT(*) AS num_schools,
    ROUND(AVG(cost_attendance), 0) AS avg_cost,
    ROUND(AVG(debt_median), 0) AS avg_debt,
    ROUND(AVG(earnings_10yr), 0) AS avg_earnings,
    ROUND(AVG(default_rate_3yr) * 100, 1) AS avg_default_pct,
    ROUND(AVG(completion_rate) * 100, 1) AS avg_completion_pct,
    ROUND(AVG(earnings_10yr) / NULLIF(AVG(debt_median), 0), 2) AS roi_ratio
FROM cost_tiers
GROUP BY cost_quartile
ORDER BY cost_quartile
""")

In [ ]:
# 📈 Visualization: Cost vs Outcomes
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

tiers = q7['cost_tier'].tolist()
colors = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']

axes[0].bar(range(4), q7['avg_cost'], color=colors)
axes[0].set_xticks(range(4))
axes[0].set_xticklabels(['Q1\nCheap', 'Q2', 'Q3', 'Q4\nExpensive'], fontsize=9)
axes[0].set_title('Avg Cost of Attendance', fontweight='bold')
axes[0].set_ylabel('$')

axes[1].bar(range(4), q7['avg_earnings'], color=colors)
axes[1].set_xticks(range(4))
axes[1].set_xticklabels(['Q1\nCheap', 'Q2', 'Q3', 'Q4\nExpensive'], fontsize=9)
axes[1].set_title('Avg Earnings (10yr)', fontweight='bold')
axes[1].set_ylabel('$')

axes[2].bar(range(4), q7['roi_ratio'], color=colors)
axes[2].set_xticks(range(4))
axes[2].set_xticklabels(['Q1\nCheap', 'Q2', 'Q3', 'Q4\nExpensive'], fontsize=9)
axes[2].set_title('ROI Ratio (Earnings/Debt)', fontweight='bold')

plt.suptitle('Does Paying More = Earning More?', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('q7_cost_vs_outcomes.png', dpi=150, bbox_inches='tight')
plt.show()

### Q8: Hidden Gems — High Earnings, Low Debt, Under the Radar
**Skills**: `Subquery` filtering, multiple conditions

**Business Value**: Actionable recommendations for students

In [ ]:
q8 = sql("""
SELECT 
    i.inst_name, i.state, i.control_label,
    i.ugds AS enrollment,
    ROUND(lo.debt_median, 0) AS debt,
    ROUND(lo.earnings_10yr, 0) AS earnings_10yr,
    ROUND(lo.earnings_10yr / lo.debt_median, 1) AS roi_ratio,
    ROUND(lo.default_rate_3yr * 100, 1) AS default_pct,
    ROUND(lo.completion_rate * 100, 1) AS completion_pct
FROM institutions i
JOIN loan_outcomes lo ON i.unitid = lo.unitid
JOIN institution_costs ic ON i.unitid = ic.unitid
WHERE 
    lo.earnings_10yr > (SELECT AVG(earnings_10yr) * 1.2 FROM loan_outcomes WHERE earnings_10yr IS NOT NULL)
    AND lo.debt_median < (SELECT AVG(debt_median) FROM loan_outcomes WHERE debt_median IS NOT NULL)
    AND lo.default_rate_3yr < 0.05
    AND lo.completion_rate > 0.50
    AND i.ugds >= 500
ORDER BY roi_ratio DESC
LIMIT 20
""")

---
## 📋 Summary Dashboard
Key numbers for your README.

In [ ]:
summary = sql("""
SELECT '🏫 Total Institutions' AS metric, CAST(COUNT(*) AS TEXT) AS value FROM institutions
UNION ALL
SELECT '📊 With Loan Data', CAST(COUNT(*) AS TEXT) FROM loan_outcomes WHERE debt_median IS NOT NULL
UNION ALL
SELECT '💰 Overall Median Debt', '$' || CAST(ROUND(AVG(debt_median), 0) AS TEXT) FROM loan_outcomes WHERE debt_median IS NOT NULL
UNION ALL
SELECT '📉 Avg Default Rate', CAST(ROUND(AVG(default_rate_3yr) * 100, 1) AS TEXT) || '%' FROM loan_outcomes WHERE default_rate_3yr IS NOT NULL
UNION ALL
SELECT '💵 Avg Earnings (10yr)', '$' || CAST(ROUND(AVG(earnings_10yr), 0) AS TEXT) FROM loan_outcomes WHERE earnings_10yr IS NOT NULL
UNION ALL
SELECT '🔴 High-Risk Schools (>15%)', CAST(COUNT(*) AS TEXT) FROM loan_outcomes WHERE default_rate_3yr > 0.15
UNION ALL
SELECT '🟢 Low-Risk Schools (<5%)', CAST(COUNT(*) AS TEXT) FROM loan_outcomes WHERE default_rate_3yr < 0.05
""")

---
## 💾 Export Results for GitHub
Run this to download all visualizations and key results.

In [ ]:
# Save key query results to CSV for GitHub
q1.to_csv('output_q1_state_ranking.csv', index=False)
q2.to_csv('output_q2_school_type.csv', index=False)
q4.to_csv('output_q4_risk_tiers.csv', index=False)
q6.to_csv('output_q6_roi_ranking.csv', index=False)
q8.to_csv('output_q8_hidden_gems.csv', index=False)
summary.to_csv('output_summary.csv', index=False)

print('✅ CSV files saved!')
print('\n📁 Files to download for GitHub:')
import glob
for f in sorted(glob.glob('output_*') + glob.glob('q*.png')):
    print(f'   {f}')

In [ ]:
# Download all outputs as ZIP
import shutil
!mkdir -p export
!cp output_*.csv export/
!cp q*.png export/ 2>/dev/null || true
shutil.make_archive('student_loan_sql_outputs', 'zip', 'export')

from google.colab import files
files.download('student_loan_sql_outputs.zip')
print('\n✅ Download started! Put these files in your GitHub repo outputs/ folder.')

---
## ✅ Next Steps

1. **Update your README**: Fill in the Key Findings section with real numbers from the summary above
2. **Add screenshots**: Put the visualization PNGs in your `outputs/` folder
3. **Push to GitHub**: `git add . && git commit -m 'Add SQL analysis results' && git push`
4. **Write insights**: The `💡 Your Insight` sections are where your analytical thinking shines — fill them in!
5. **Link to Part 1**: Make sure both repos link to each other

---
*Author: YUN TING SU (Katherine) | kt0704@bu.edu*